# Bloque 3: Ensamblando el Agente con LangGraph

Hasta ahora, tenemos un "Cerebro" (el LLM) y unas "Manos" (las Herramientas). LangGraph es el "Sistema Nervioso" que los conecta.

LangGraph nos permite construir agentes definiendo un **Grafo de Estado** (StateGraph) donde:
1. **El Estado (State):** Es la memoria compartida (en nuestro caso, el historial de mensajes).
2. **Los Nodos (Nodes):** Son funciones de Python que hacen el trabajo (ej. pensar, ejecutar una herramienta).
3. **Las Aristas (Edges):** Son las reglas que deciden qué nodo se ejecuta después (flujo de control).

Implementaremos la arquitectura **ReAct (Reason + Act)**, creando un ciclo donde el modelo razona, decide usar una herramienta, observa el resultado, y vuelve a razonar.

In [ ]:
import os
import sys
from dotenv import load_dotenv

# Agregamos la ruta padre para poder importar desde la carpeta 'app'
sys.path.append(os.path.abspath('..'))
load_dotenv(dotenv_path="../.env")

from langchain_google_genai import ChatGoogleGenerativeAI
# Importamos las herramientas que ya programamos
from app.tools import buscar_cliente, obtener_precio_accion, consultar_inventario

# 1. Agrupamos las herramientas y configuramos el modelo
tools = [buscar_cliente, obtener_precio_accion, consultar_inventario]
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
llm_with_tools = llm.bind_tools(tools)

print("✅ Modelo y herramientas configuradas.")

## 1. Definiendo el Estado y los Nodos

Utilizaremos `MessagesState`, un estado predefinido por LangGraph que básicamente es un diccionario con una clave `"messages"` que contiene la lista de la conversación.

Luego, definiremos nuestros dos nodos principales:
* **Nodo Agente:** Llama al LLM para que evalúe el estado y decida qué hacer.
* **Nodo Herramientas:** Toma la petición del LLM, ejecuta la función de Python correspondiente y devuelve el resultado.

In [ ]:
from langgraph.graph import MessagesState
from langgraph.prebuilt import ToolNode
from langchain_core.messages import SystemMessage

# Nodo 1: El cerebro del agente
def chatbot_node(state: MessagesState):
    """
    Este nodo es el 'Cerebro'. Toma el historial de mensajes actual, 
    le añade instrucciones de sistema y le pide al LLM que decida el siguiente paso.
    """
    mensajes = [
        SystemMessage(content="""Eres un asistente corporativo altamente eficiente. 
        Tienes acceso a bases de datos de inventario, clientes, y herramientas de agenda.
        Usa las herramientas a tu disposición para responder a las solicitudes del usuario.
        Si no sabes la respuesta o la herramienta falla, admítelo y pide aclaraciones.
        La fecha actual es 26 de febrero de 2026.""")
    ] + state["messages"]
    
    # Invocamos al modelo
    response = llm_with_tools.invoke(mensajes)
    
    # Devolvemos el nuevo mensaje generado (LangGraph lo agregará a la lista automáticamente)
    return {"messages": [response]}

# Nodo 2: El ejecutor de herramientas
# LangGraph ya tiene un 'ToolNode' preconstruido que hace exactamente lo que necesitamos
tool_node = ToolNode(tools=tools)

print("✅ Nodos definidos.")

## 2. Construyendo el Grafo y las Aristas (Edges)

Aquí es donde definimos el flujo del programa. Usaremos una "Arista Condicional" (`conditional_edge`) para que el grafo decida automáticamente si debe ir a las herramientas (si el LLM lo pidió) o si debe terminar (si el LLM dio una respuesta final).

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import tools_condition

# Inicializamos el grafo con nuestro tipo de estado
workflow = StateGraph(MessagesState)

# Agregamos los nodos al grafo
workflow.add_node("agent", chatbot_node)
workflow.add_node("tools", tool_node)

# Definimos el flujo:
# 1. Siempre empezamos en el agente
workflow.add_edge(START, "agent")

# 2. Después del agente, evaluamos la condición:
# 'tools_condition' revisa si el mensaje generado tiene 'tool_calls'.
# Si SÍ tiene -> va al nodo "herramientas"
# Si NO tiene -> va a "END" (termina la ejecución)
workflow.add_conditional_edges("agent", tools_condition)

# 3. Después de ejecutar una herramienta, SIEMPRE regresamos al agente
# para que evalúe la nueva información
workflow.add_edge("tools", "agent")

# Compilamos el grafo para que esté listo para ejecutarse
graph = workflow.compile()

print("✅ Grafo compilado exitosamente.")

## 3. Poniendo al Agente a Trabajar

Vamos a hacerle una pregunta al agente que lo obligue a usar la herramienta de base de datos que programamos, específicamente para ver cómo maneja el problema de **Ambigüedad** con el nombre "Carlos".

Usaremos el método `.stream()` para ver cómo la información fluye de nodo a nodo en tiempo real.

In [ ]:
from langchain_core.messages import HumanMessage

# La petición inicial del usuario
mensaje = HumanMessage(content="¿Me puedes dar el correo de Carlos?")

print("Iniciando ejecución del grafo...\n")
print("-" * 40)

# stream() nos devuelve actualizaciones de estado cada vez que un nodo termina
for event in graph.stream({"messages": [mensaje]}):
    for node_name, node_state in event.items():
        print(f"--- [Nodo ejecutado: {node_name}] ---")
        
        # Obtenemos el último mensaje generado por este nodo
        ultimo_mensaje = node_state["messages"][-1]
        
        # Si el nodo es el agente y generó una llamada a herramienta:
        if node_name == "agent" and hasattr(ultimo_mensaje, "tool_calls") and ultimo_mensaje.tool_calls:
            for tool_call in ultimo_mensaje.tool_calls:
                print(f"🔧 Herramienta solicitada: {tool_call['name']}")
                print(f"📥 Argumentos: {tool_call['args']}")
                
        # Si el nodo es el agente y dio una respuesta final (texto):
        elif node_name == "agent" and ultimo_mensaje.content:
            print(f"\n🤖 Agente: {ultimo_mensaje.content}\n")
            
        # Si el nodo son las herramientas, mostramos el resultado crudo:
        elif node_name == "tools":
            print(f"📤 Resultado de la herramienta: {ultimo_mensaje.content[:150]}...\n")
        
        print("-" * 40)